In [1]:
import pandas as pd
import numpy as np
import joblib
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score, f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

c:\Users\Buwaneka Fernando\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
tokenizer     = AutoTokenizer.from_pretrained("models/roberta_checkpoint")
roberta       = AutoModelForSequenceClassification.from_pretrained(
    "models/roberta_checkpoint"
)
roberta = roberta.to(device); roberta.eval()

In [ ]:
demo_model   = joblib.load("models/demographic_classifier.pkl")
feature_cols = joblib.load("models/demographic_feature_cols.pkl")

In [ ]:
product_data = pd.read_csv("data/final/train.csv")
demo_data    = pd.read_csv("data/processed/demographic_features.csv")

In [ ]:
print(f"Product training data: {len(product_data)} rows")
print(f"Demographic data:      {len(demo_data)} rows (survey responses)")

In [ ]:
# function to get RoBERTa probabilities
def get_roberta_probs(texts, batch_size=32):
    all_probs = []
    for i in range(0, len(texts), batch_size):
        batch = list(texts[i:i+batch_size])
        enc   = tokenizer(batch, max_length=256, padding=True,
                          truncation=True, return_tensors='pt').to(device)
        with torch.no_grad():
            out   = roberta(**enc)
            probs = torch.softmax(out.logits, dim=1)
        all_probs.extend(probs.cpu().numpy())
        if i % 500 == 0:
            print(f"  RoBERTa: {min(i+batch_size, len(texts))}/{len(texts)}", end='\r')
    print(f"\n  RoBERTa done: {len(texts)} samples")
    return np.array(all_probs)

In [ ]:
# get RoBERTa probabilities for a sample of the product data
sample_n = min(500, len(product_data))
product_sample = product_data.sample(sample_n, random_state=42).reset_index(drop=True)
 
print(f"\nGetting RoBERTa probs for {sample_n} product samples...")
roberta_probs = get_roberta_probs(product_sample['input_text'].tolist())
y_product     = product_sample['cognitive_label'].values

In [ ]:
# get demographic model probabilities for the demo data
demo_X     = demo_data[feature_cols].values
demo_probs = demo_model.predict_proba(demo_X)   # shape: (20, 2)
y_demo     = demo_data['cognitive_label'].values

In [ ]:
print(f"\nDemographic probs shape: {demo_probs.shape}")
print(f"Demo label distribution: S1={sum(y_demo==1)} S2={sum(y_demo==0)}")

In [ ]:
# create fusion training data by matching demo rows to product rows with the same label
fusion_X = []
fusion_y = []
 
for i in range(len(demo_data)):
    demo_label = y_demo[i]
    dp = demo_probs[i]   # demographic probabilities for this person
 
    # Find product samples with same label
    matching_product_idx = np.where(y_product == demo_label)[0]
    if len(matching_product_idx) == 0:
        continue
 
    # Sample 3 product matches per demographic row
    chosen = np.random.RandomState(i).choice(
        matching_product_idx, size=min(3, len(matching_product_idx)), replace=False
    )
    for pidx in chosen:
        rp = roberta_probs[pidx]
        combined = np.concatenate([rp, dp])   
        fusion_X.append(combined)
        fusion_y.append(demo_label)
 
fusion_X = np.array(fusion_X)
fusion_y = np.array(fusion_y)
 
print(f"\nFusion training samples: {len(fusion_X)}")
print(f"Fusion labels: S1={sum(fusion_y==1)} S2={sum(fusion_y==0)}")

In [ ]:
meta_model = LogisticRegression(
    C=1.0, class_weight='balanced',
    max_iter=1000, random_state=42
)
meta_model.fit(fusion_X, fusion_y)
 
# Evaluate with LOO-CV on fusion data
from sklearn.model_selection import cross_val_score
loo    = LeaveOneOut()
loo_cv = cross_val_score(meta_model, fusion_X, fusion_y,
                          cv=loo, scoring='f1_weighted')
 
print(f"\nMeta-learner LOO-CV F1: {loo_cv.mean():.4f} ± {loo_cv.std():.4f}")

In [ ]:
test_data   = pd.read_csv("data/final/test.csv")
test_probs  = get_roberta_probs(test_data['input_text'].tolist())
y_test      = test_data['cognitive_label'].values

In [ ]:
roberta_preds = np.argmax(test_probs, axis=1)
roba_acc = accuracy_score(y_test, roberta_preds)
roba_f1  = f1_score(y_test, roberta_preds, average='weighted')

In [ ]:
print(f"RoBERTa only:  Acc={roba_acc:.4f}  F1={roba_f1:.4f}")

In [ ]:
# Save model and config
joblib.dump(meta_model, "models/fusion_meta_model.pkl")
 
fusion_config = {
    'product_weight':     0.70,
    'demographic_weight': 0.30,
    'use_meta_learner':   True,
    'feature_cols':       feature_cols,
    'training_samples':   len(demo_data),
    'note': 'Meta-learner trained on matched product+demographic pairs'
}
joblib.dump(fusion_config, "models/fusion_config.pkl")